# 04 - Alert: high admissions -> email

Reads `patients_last_hour_gold` and, if the number of patients received in the last hour reaches `THRESHOLD`,
calls the **Azure Logic App** (HTTP trigger) which sends the email.

In [ ]:
import requests

SCHEMA = "depi.hospitals"
THRESHOLD = 100   # the producer sends ~180 events/hour (one every 10-30 s) -> lower/raise this to test the alert

SECRET_SCOPE = "hospital-scope"   # Databricks secret scope (see README -> "Secrets")

def get_secret(name):
    """Read a secret from the Databricks secret scope.
    Fallback: a notebook widget (paste the value at runtime - clear it before committing the notebook)."""
    try:
        return dbutils.secrets.get(scope=SECRET_SCOPE, key=name)
    except Exception:
        dbutils.widgets.text(name, "")
        value = dbutils.widgets.get(name)
        if not value:
            raise ValueError(f"Secret '{name}' not found in scope '{SECRET_SCOPE}' and widget '{name}' is empty.")
        return value

In [ ]:
num_patients = spark.table(f"{SCHEMA}.patients_last_hour_gold").collect()[0]["num_patients_last_hour"]
print("Patients in the last hour:", num_patients)

if num_patients >= THRESHOLD:
    response = requests.post(
        get_secret("logic-app-url"),
        json={"num_patients_last_hour": int(num_patients)},   # key name must match docs/logic_app_email_template.txt
        timeout=30,
    )
    response.raise_for_status()
    print("Alert sent! HTTP", response.status_code)
else:
    print(f"No alert needed (threshold = {THRESHOLD})")